# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/franciskendrick/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Anatomy of Growing Content

The paper compares pages labelled **up** versus **down** using `trend_direction`, calculated from a 30-day-versus-previous-30-day impression change. It reports that the growing cohort is younger and longer on average, while explicitly describing the comparison as observational.

**Methodology question:** the label is a contemporaneous trend category, not a future outcome. Does the evidence support the descriptive statement that these cohorts differ? Yes, as an observed portfolio comparison. Does it support the stronger operational implication that expanding a thin page or refreshing an aging page will *cause* future growth? Not by itself. A stronger test would define features before $t$, define a future outcome after $t$, and evaluate whether the proposed action predicts or improves that future outcome.

**Constructive review:** the paper already protects the finding with observational language and recommends measuring impressions, clicks, and position after an update. I would preserve that distinction explicitly: the cohort comparison is evidence of an association; the recommended refresh action is a decision-support hypothesis requiring prospective or experimental validation.

### Finding 2 — The Content Performance Curve

The paper groups content by **content age** and compares the resulting groups using FlyRank's composite **health score**. It reports a peak around 61–90 days, a lower value around 271–365 days, and a higher 365+ value that is interpreted narrowly because refreshed older pages and survivor effects complicate the comparison.

**Methodology question:** where exactly does the outcome come from, and how much of it is mechanically related to the explanatory variables? Health score is a composite of impressions, position, CTR, and scroll depth, so it is an observed portfolio context metric rather than an independent causal outcome. Age is also entangled with publication cohort, topic, client mix, accumulated history, and refresh selection. Does an age-bucket comparison validate a lifecycle effect, or does it mainly describe differences among cohorts that survived into each bucket?

**Constructive review:** the paper already acknowledges strong survivor bias in the 365+ cell and that age alone does not prove natural recovery. The safest interpretation is that performance differs across observed age cohorts in this snapshot. A longitudinal or time-to-event design would be needed to establish a genuine lifecycle effect or estimate the effect of refreshing at a particular age.

### Audit principle

> A finding can be statistically descriptive and still be useful without being causal. The validation design has to match the verb in the claim: **observed** and **associated** require less evidence than **predicts**, **improves**, or **causes**.


In [1]:
paper_audit = [
    {
        "finding": "Finding #1 — Anatomy of Growing Content",
        "label_source": "30d-vs-previous-30d impression trend_direction",
        "evidence_type": "Observational cohort comparison",
        "safe_claim": "Growing and declining cohorts differed in observed age/depth profiles",
        "not_established": "Expanding or refreshing content causes future growth",
    },
    {
        "finding": "Finding #2 — Content Performance Cur  ve",
        "label_source": "Observed health score grouped by content age",
        "evidence_type": "Cross-sectional age-bucket comparison",
        "safe_claim": "Observed health differed across age cohorts in this snapshot",
        "not_established": "Age alone causes decline or refresh causes recovery",
    },
]

import pandas as pd
print(pd.DataFrame(paper_audit).to_string(index=False))

                                 finding                                   label_source                         evidence_type                                                            safe_claim                                      not_established
 Finding #1 — Anatomy of Growing Content 30d-vs-previous-30d impression trend_direction       Observational cohort comparison Growing and declining cohorts differed in observed age/depth profiles Expanding or refreshing content causes future growth
Finding #2 — Content Performance Cur  ve   Observed health score grouped by content age Cross-sectional age-bucket comparison          Observed health differed across age cohorts in this snapshot  Age alone causes decline or refresh causes recovery


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### What this audit found in Week 5

The Week-5 **prose** described a grouped temporal split, but the executable model cell actually used `train_test_split(..., stratify=y)`. That is a row-level random split, not a client holdout. The audit therefore treats the Week-5 result as the **before** condition and re-runs the same model under a client-grouped split as the **after** condition.

There is a second implementation issue: Week 5 loaded only March and calculated a 7-day forward target. Decision dates near the end of March therefore cannot observe a complete target window. This audit uses February–April partition globs but restricts decision dates to March 1–24, so every retained row has a complete seven-day forward window while still using the March decision period.

The model and feature definitions remain close to Week 5: Random Forest, the same six feature columns, the same future-click decline proxy, and **Precision@50** as the operational ranking metric. The methodological change being tested is the validation split.

In [3]:
import os
import numpy as np
import pandas as pd
import duckdb

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score

# ============================================================
# 1. DuckDB + Hugging Face authentication
# ============================================================

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN is required. Add it to Colab userdata or the environment."
    )

con = duckdb.connect()

safe_token = hf_token.replace("'", "''")

con.execute(
    f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{safe_token}');"
)

# ============================================================
# 2. Partition-pruned warehouse paths
# ============================================================

WAREHOUSE_URI = "hf://datasets/FlyRank/internship-warehouse"

# We need:
# - February 22 onward for the 7-day historical lookback
# - March through March 30 for the 7-day forward target
#
# March 1–24 are the decision dates retained for the audit.
FACT_FEB_PATH = (
    f"{WAREHOUSE_URI}/fact_content_daily_performance/"
    f"month=2026-02/*.parquet"
)

FACT_MAR_PATH = (
    f"{WAREHOUSE_URI}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)

DIM_CONTENT_PATH = f"{WAREHOUSE_URI}/dim_content.parquet"

# IMPORTANT:
# Do NOT construct a "{path1,path2}" glob.
# Instead, UNION the individually pruned partition paths.
fact_union = f"""
    SELECT *
    FROM read_parquet('{FACT_FEB_PATH}', union_by_name=true)

    UNION ALL

    SELECT *
    FROM read_parquet('{FACT_MAR_PATH}', union_by_name=true)
"""

# ============================================================
# 3. Build the audit dataset
# ============================================================

query = f"""
WITH fact AS (
    {fact_union}
),

base AS (
    SELECT
        f.report_date,
        f.client_hash_id,
        f.content_hash_id,

        c.word_count,
        c.search_volume,
        c.competition,
        c.cpc,

        -- Historical features:
        -- strictly t-7 through t-1
        AVG(f.gsc_clicks) OVER (
            PARTITION BY f.client_hash_id, f.content_hash_id
            ORDER BY f.report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS hist_7d_avg_clicks,

        SUM(f.ga4_total_engagement_sec) OVER (
            PARTITION BY f.client_hash_id, f.content_hash_id
            ORDER BY f.report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS hist_7d_sum_engagement,

        COUNT(*) OVER (
            PARTITION BY f.client_hash_id, f.content_hash_id
            ORDER BY f.report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS hist_7d_rows,

        -- Future label:
        -- t through t+6
        SUM(f.gsc_clicks) OVER (
            PARTITION BY f.client_hash_id, f.content_hash_id
            ORDER BY f.report_date
            ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
        ) AS target_future_clicks_7d,

        COUNT(*) OVER (
            PARTITION BY f.client_hash_id, f.content_hash_id
            ORDER BY f.report_date
            ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
        ) AS future_rows_7d

    FROM fact f

    LEFT JOIN read_parquet('{DIM_CONTENT_PATH}') c
        ON f.client_hash_id = c.client_hash_id
       AND f.content_hash_id = c.content_hash_id

    WHERE f.report_date BETWEEN DATE '2026-02-22'
                            AND DATE '2026-03-30'

      AND f.gsc_data_available IS TRUE
      AND f.ga4_data_available IS TRUE
)

SELECT *
FROM base

WHERE report_date BETWEEN DATE '2026-03-01'
                      AND DATE '2026-03-24'

  -- Require a complete 7-row historical window.
  AND hist_7d_rows = 7

  -- Require a complete t through t+6 future window.
  AND future_rows_7d = 7
"""

df_clean = con.sql(query).df()

# ============================================================
# 4. Define features and future target
# ============================================================

feature_cols = [
    "word_count",
    "search_volume",
    "competition",
    "cpc",
    "hist_7d_avg_clicks",
    "hist_7d_sum_engagement",
]

target_col = "target_decline"

# Ensure numeric model inputs.
for col in feature_cols:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
    )

# Missing metadata is represented as zero,
# matching the Week-5 treatment.
df_clean[feature_cols] = df_clean[feature_cols].fillna(0)

# Future decline proxy:
#
# future 7-day clicks < historical average daily clicks * 7
#
# Historical features are t-7 through t-1.
# Target is t through t+6.
df_clean[target_col] = (
    df_clean["target_future_clicks_7d"]
    <
    df_clean["hist_7d_avg_clicks"] * 7
).astype(int)

# Remove invalid rows.
df_clean = (
    df_clean
    .replace([np.inf, -np.inf], np.nan)
    .dropna(
        subset=feature_cols + [
            target_col,
            "client_hash_id",
            "content_hash_id",
        ]
    )
)

if len(df_clean) == 0:
    raise RuntimeError(
        "The audit query returned zero valid rows."
    )

if df_clean[target_col].nunique() < 2:
    raise RuntimeError(
        "The retained audit slice contains only one target class."
    )

# ============================================================
# 5. Dataset diagnostics
# ============================================================

print("--- AUDIT DATASET ---")
print(f"Rows              : {len(df_clean):,}")
print(
    f"Clients           : "
    f"{df_clean['client_hash_id'].nunique():,}"
)
print(
    f"Content entities  : "
    f"{df_clean['content_hash_id'].nunique():,}"
)
print(
    f"Decision dates    : "
    f"{df_clean['report_date'].min()} "
    f"to "
    f"{df_clean['report_date'].max()}"
)
print(
    f"Positive rate     : "
    f"{df_clean[target_col].mean():.4f}"
)

print("\n--- TEMPORAL CHECKS ---")
print(
    "Historical rows = 7 :",
    df_clean["hist_7d_rows"].eq(7).all()
)
print(
    "Future rows = 7     :",
    df_clean["future_rows_7d"].eq(7).all()
)

print("\n--- FEATURE SET ---")
print(", ".join(feature_cols))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- AUDIT DATASET ---
Rows              : 67,559
Clients           : 23
Content entities  : 8,245
Decision dates    : 2026-03-01 00:00:00 to 2026-03-24 00:00:00
Positive rate     : 0.4833

--- TEMPORAL CHECKS ---
Historical rows = 7 : True
Future rows = 7     : True

--- FEATURE SET ---
word_count, search_volume, competition, cpc, hist_7d_avg_clicks, hist_7d_sum_engagement


In [4]:
# ============================================================
# 6. Model + evaluation helper
# ============================================================

def fit_and_rank(
    X_train,
    X_test,
    y_train,
    y_test,
    test_rows,
    split_name
):
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)

    if 1 in model.classes_:
        positive_idx = list(model.classes_).index(1)
        prob_decline = probabilities[:, positive_idx]
    else:
        prob_decline = np.zeros(len(X_test))

    ranked = test_rows.copy()

    ranked["prob_decline"] = prob_decline

    ranked = ranked.sort_values(
        "prob_decline",
        ascending=False
    ).reset_index(drop=True)

    # Precision@50 is defined as the observed positive rate
    # among the 50 highest-ranked rows.
    k = min(50, len(ranked))

    top_k = ranked.head(k)

    precision_at_k = (
        top_k[target_col].mean()
        if k > 0
        else np.nan
    )

    average_precision = average_precision_score(
        y_test,
        prob_decline
    )

    if len(np.unique(y_test)) == 2:
        roc_auc = roc_auc_score(
            y_test,
            prob_decline
        )
    else:
        roc_auc = np.nan

    return {
        "split": split_name,
        "test_rows": len(y_test),
        "test_clients": (
            test_rows["client_hash_id"].nunique()
        ),
        "ROC_AUC": roc_auc,
        "Average_Precision": average_precision,
        "Precision@50": precision_at_k,
        "ranked": ranked,
        "model": model,
    }


# ============================================================
# 7. Prepare X and y
# ============================================================

X = df_clean[feature_cols]
y = df_clean[target_col]

# ============================================================
# 8. BEFORE — reproduce Week-5 row-level random split
# ============================================================

indices = np.arange(len(df_clean))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

before = fit_and_rank(
    X.iloc[train_idx],
    X.iloc[test_idx],
    y.iloc[train_idx],
    y.iloc[test_idx],
    df_clean.iloc[test_idx],
    "Week-5 row-level random split",
)

# ============================================================
# 9. AFTER — honest client-grouped split
# ============================================================

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42,
)

group_train_idx, group_test_idx = next(
    group_splitter.split(
        X,
        y,
        groups=df_clean["client_hash_id"],
    )
)

after = fit_and_rank(
    X.iloc[group_train_idx],
    X.iloc[group_test_idx],
    y.iloc[group_train_idx],
    y.iloc[group_test_idx],
    df_clean.iloc[group_test_idx],
    "Honest client-group holdout",
)

# ============================================================
# 10. Before / after comparison
# ============================================================

comparison = pd.DataFrame([
    {
        "split": before["split"],
        "test_rows": before["test_rows"],
        "test_clients": before["test_clients"],
        "ROC_AUC": before["ROC_AUC"],
        "Average_Precision": before["Average_Precision"],
        "Precision@50": before["Precision@50"],
    },
    {
        "split": after["split"],
        "test_rows": after["test_rows"],
        "test_clients": after["test_clients"],
        "ROC_AUC": after["ROC_AUC"],
        "Average_Precision": after["Average_Precision"],
        "Precision@50": after["Precision@50"],
    },
])

print("--- BEFORE / AFTER VALIDATION ---")

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

precision_delta = (
    after["Precision@50"]
    -
    before["Precision@50"]
)

print(
    f"\nPrecision@50 change: "
    f"{precision_delta:+.4f}"
)

--- BEFORE / AFTER VALIDATION ---
                        split  test_rows  test_clients  ROC_AUC  Average_Precision  Precision@50
Week-5 row-level random split      20268            22   0.6705             0.6201        0.9000
  Honest client-group holdout       1135             7   0.7171             0.5504        0.4600

Precision@50 change: -0.4400


### What the before/after test means

The key comparison is **not** whether the grouped score is larger. The methodological question is whether performance remains credible when entire clients are withheld. A random row split allows pages from the same client to appear in both training and test, so client-specific distributions can make the task easier than the production scenario.

If **Precision@50 falls materially** under the client holdout, the safe conclusion is that the Week-5 ranking was partly dependent on information shared within clients. If it remains similar, that is evidence of better transfer across unseen clients, but it still does not establish causal impact or guarantee future performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The audit treats leakage as a column-level and window-level problem. Every feature must be knowable at $t$, while the label is calculated from the forward window.

1. **Split leakage:** the executable Week-5 code used a row-level random split even though the prose described grouped validation.
2. **Entity-history leakage:** Week 5's historical windows were partitioned by `content_hash_id` only; the audit partitions by `(client_hash_id, content_hash_id)` so histories cannot cross client boundaries.
3. **Future-window completeness:** a March-only scan cannot fully observe seven days after late-March decision dates. The audit restricts decisions to March 1–24 and scans the required surrounding partitions.
4. **Target-derived fields:** `target_future_clicks_7d` and `future_rows_7d` are explicitly excluded from `feature_cols`.
5. **Product-decision leakage:** no `health_score`, `priority_score`, `action_type`, or optimization flag is used as a model feature. The internship guide explicitly warns against feeding reconstructed product decisions back into discovery models.

In [7]:
leakage_audit = pd.DataFrame([
    ["report_date", "Context / decision time", "Defines the decision period", "PASS"],
    ["word_count", "Feature", "Content metadata; no future performance calculation", "PASS*"],
    ["search_volume", "Feature", "Content metadata; treated as a pre-decision signal", "PASS*"],
    ["competition", "Feature", "Content metadata; treated as a pre-decision signal", "PASS*"],
    ["cpc", "Feature", "Content metadata; treated as a pre-decision signal", "PASS*"],
    ["hist_7d_avg_clicks", "Feature", "Rows t-7 through t-1 only", "PASS"],
    ["hist_7d_sum_engagement", "Feature", "Rows t-7 through t-1 only", "PASS"],
    ["target_future_clicks_7d", "Label only", "Contains t through t+6", "EXCLUDED FROM FEATURES"],
    ["future_rows_7d", "Validation guard", "Ensures complete future label window", "EXCLUDED FROM FEATURES"],
    ["client_hash_id", "Group key", "Used for client holdout, not as a predictive feature", "PASS"],
    ["content_hash_id", "Join / history key", "Used for entity history, not as a predictive feature", "PASS"],
    ["health_score / priority_score / action_type", "Product decision outputs", "Not present in feature set", "EXCLUDED"],
], columns=["field", "bucket", "audit_reason", "status"])

print(leakage_audit.to_string(index=False))

forbidden = {
    "target_decline", "target_future_clicks_7d", "future_rows_7d",
    "health_score", "priority_score", "action_type"
}
violations = sorted(set(feature_cols).intersection(forbidden))
assert not violations, f"Leakage violation in feature set: {violations}"
assert df_clean["future_rows_7d"].eq(7).all(), "Incomplete future windows remain."

print("\nAutomated leakage checks: PASS")
print(f"Feature columns audited: {', '.join(feature_cols)}")

                                      field                   bucket                                         audit_reason                 status
                                report_date  Context / decision time                          Defines the decision period                   PASS
                                 word_count                  Feature  Content metadata; no future performance calculation                  PASS*
                              search_volume                  Feature   Content metadata; treated as a pre-decision signal                  PASS*
                                competition                  Feature   Content metadata; treated as a pre-decision signal                  PASS*
                                        cpc                  Feature   Content metadata; treated as a pre-decision signal                  PASS*
                         hist_7d_avg_clicks                  Feature                            Rows t-7 through t-1 only         

In [5]:
# Real failure examples from the honest client-held-out ranking.
# Only pseudonymized IDs and safe numeric signals are displayed.
honest_ranked = after["ranked"].copy()
false_positives = honest_ranked[honest_ranked[target_col] == 0].head(10).copy()

# Lowest-ranked positive examples are useful false-negative cases.
false_negatives = honest_ranked.sort_values("prob_decline", ascending=True)
false_negatives = false_negatives[false_negatives[target_col] == 1].head(10).copy()

failure_cols = [
    "client_hash_id", "content_hash_id", "report_date",
    "prob_decline", target_col
] + feature_cols

print("--- FALSE POSITIVES: HIGH-RANKED BUT NOT LABEL-POSITIVE ---")
print(
    false_positives[failure_cols].to_string(index=False)
    if len(false_positives)
    else "No displayed false positives."
)

print("\n--- FALSE NEGATIVES: LABEL-POSITIVE BUT RANKED LOW ---")
print(
    false_negatives[failure_cols].to_string(index=False)
    if len(false_negatives)
    else "No displayed false negatives."
)

print("\nThese are errors against the proxy label, not proof that a page was commercially good or bad.")

--- FALSE POSITIVES: HIGH-RANKED BUT NOT LABEL-POSITIVE ---
         client_hash_id          content_hash_id report_date  prob_decline  target_decline  word_count  search_volume  competition  cpc  hist_7d_avg_clicks  hist_7d_sum_engagement
client_b10cb2997d0c7c86 content_06dcc81394435cf5  2026-03-17      0.613233               0        4561              0          0.0  0.0            1.857143                   730.0
client_b10cb2997d0c7c86 content_06dcc81394435cf5  2026-03-15      0.594926               0        4561              0          0.0  0.0            1.571429                   730.0
client_b10cb2997d0c7c86 content_06dcc81394435cf5  2026-03-16      0.594926               0        4561              0          0.0  0.0            1.571429                   730.0
client_b10cb2997d0c7c86 content_06dcc81394435cf5  2026-03-14      0.592806               0        4561              0          0.0  0.0            1.428571                   730.0
client_3f0ce4d44fe94f3d content_fbd410a0

### Failure interpretation

The examples above are intentionally inspected as **proxy-label errors**. A false positive means the model ranked a page highly for predicted decline but the defined seven-day click comparison did not mark it as declining. A false negative means the opposite.

The Week-5 narrative said false positives typically reflected high historical clicks and competition. That is too strong without a broader error sample and a controlled comparison. The executable audit therefore measures the actual examples first and treats any feature pattern as **directional** rather than universal.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original Week-5-style claim

> “The Random Forest identifies pages that are on the precipice of structural decline and provides the right pages to fix.”

### Audited rewrite

> **On the audited March decision slice, the Random Forest produced a ranked list of pages using observed historical signals. Under a client-grouped holdout, Precision@50 measures how often the highest-ranked pages matched the defined seven-day decline proxy. This is directional decision-support for review prioritization; it does not establish that a page will decline, that a refresh will recover it, or that the model transfers unchanged to future periods or unseen data distributions.**

### What changed

| Original implication | Audited language |
|---|---|
| “identifies pages on the precipice” | “ranked pages using observed historical signals” |
| “right pages to fix” | “pages to prioritize for review” |
| implied future certainty | “matched the defined seven-day decline proxy” |
| implied causal intervention | explicitly says refresh recovery is not established |
| implied generalization | explicitly limits evidence to the audited slice and split |

The corrected claim is narrower, but it is more defensible because the model output, label, validation design, and operational decision are separated.

In [6]:
safe_terms = ["observed", "measured", "directional", "decision-support", "proxy"]
unsafe_terms = ["proves", "guarantees", "causes", "right pages to fix", "Google algorithm factor"]

rewritten_claim = (
    "The Random Forest produced a ranked list using observed historical signals. "
    "Under a client-grouped holdout, Precision@50 measures agreement with the defined "
    "seven-day decline proxy. This is directional decision-support for review "
    "prioritization, not proof of future decline or causal recovery."
)

print("--- FINAL PUBLIC-SAFE CLAIM ---")
print(rewritten_claim)
print("\nUnsafe vocabulary present:",
      [term for term in unsafe_terms if term.lower() in rewritten_claim.lower()])
assert not any(term.lower() in rewritten_claim.lower() for term in unsafe_terms)
print("Claim language audit: PASS")

--- FINAL PUBLIC-SAFE CLAIM ---
The Random Forest produced a ranked list using observed historical signals. Under a client-grouped holdout, Precision@50 measures agreement with the defined seven-day decline proxy. This is directional decision-support for review prioritization, not proof of future decline or causal recovery.

Unsafe vocabulary present: []
Claim language audit: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.